In [1]:
!pip install -q phonemizer soundfile torchaudio
!sudo apt-get update -qq && sudo apt-get install -y -qq espeak-ng

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 5.)
debconf: falling back to frontend: Readline
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 130198 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.2-2build3_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.2-2build3) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-13build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-13build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.51+dfsg-12build1_amd64.deb ...
Unpacki

In [2]:
import os
import glob
import random
import torchaudio
import soundfile as sf
from phonemizer.backend import EspeakBackend

SPEAKER_ID = "p251"
OUT_WAV_DIR = "/kaggle/working/StyleTTS2/Data/wavs"
OUT_DATA_DIR = "/kaggle/working/StyleTTS2/Data"
os.makedirs(OUT_WAV_DIR, exist_ok=True)

# 1. Find all audio files (.wav or .flac) for SPEAKER_ID inside /kaggle/input
all_speaker_files = glob.glob(f"/kaggle/input/**/{SPEAKER_ID}/**/*", recursive=True)
audio_exts = (".wav", ".flac")

# Prefer mic1 if available, otherwise take all audio files for this speaker
mic1_files = sorted([f for f in all_speaker_files if f.lower().endswith(audio_exts) and "_mic1" in os.path.basename(f)])
audio_files = mic1_files if mic1_files else sorted([f for f in all_speaker_files if f.lower().endswith(audio_exts)])

# 2. Index all .txt transcript files for SPEAKER_ID by utterance ID (e.g., p251_001)
txt_files = glob.glob(f"/kaggle/input/**/{SPEAKER_ID}/**/*.txt", recursive=True)
txt_map = {}
for t_path in txt_files:
    utt_id = os.path.splitext(os.path.basename(t_path))[0]
    utt_id = "_".join(utt_id.split("_")[:2])
    txt_map[utt_id] = t_path

print(f"Found {len(audio_files)} audio files and {len(txt_map)} transcript files for {SPEAKER_ID}.")

phonemizer = EspeakBackend(language="en-us", preserve_punctuation=True, with_stress=True)
entries = []

for audio_path in audio_files:
    fname = os.path.basename(audio_path)
    utt_id = "_".join(os.path.splitext(fname)[0].split("_")[:2])
    txt_path = txt_map.get(utt_id)
    if not txt_path:
        continue

    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    if not text:
        continue

    wav, sr = torchaudio.load(audio_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000:
        wav = torchaudio.functional.resample(wav, sr, 24000)

    out_wav_name = f"{utt_id}.wav"
    out_wav_path = os.path.join(OUT_WAV_DIR, out_wav_name)
    sf.write(out_wav_path, wav.squeeze().numpy(), 24000, subtype="PCM_16")

    phonemes = phonemizer.phonemize([text], strip=True)[0]
    entries.append(f"{out_wav_name}|{phonemes}|0")

if not entries:
    # Print sample paths if still 0 so we can see how this dataset is structured
    sample_paths = glob.glob("/kaggle/input/**/*p251*", recursive=True)[:15]
    raise RuntimeError(f"0 matched clips for {SPEAKER_ID}. Sample matching paths: {sample_paths}")

random.seed(42)
random.shuffle(entries)
split_idx = int(len(entries) * 0.9)

with open(f"{OUT_DATA_DIR}/train_list.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(entries[:split_idx]) + "\n")

with open(f"{OUT_DATA_DIR}/val_list.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(entries[split_idx:]) + "\n")

print(f"Prepared {len(entries)} clips for {SPEAKER_ID}: {split_idx} train, {len(entries) - split_idx} val.")

Found 364 audio files and 364 transcript files for p251.
Prepared 364 clips for p251: 327 train, 37 val.


In [3]:
%cd /kaggle/working
!git clone https://github.com/yl4579/StyleTTS2.git
%cd /kaggle/working/StyleTTS2
!pip install -q munch pydub pyyaml librosa nltk matplotlib accelerate transformers einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git
!git lfs install && git clone https://huggingface.co/yl4579/StyleTTS2-LibriTTS
!cp -r StyleTTS2-LibriTTS/Models/* Models/

/kaggle/working
fatal: destination path 'StyleTTS2' already exists and is not an empty directory.
/kaggle/working/StyleTTS2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	log
cp: cannot stat 'StyleTTS2-LibriTTS/Models/*': No such file or directory


In [4]:
%cd /kaggle/working
!pip install -q huggingface_hub
!git clone --no-checkout https://github.com/yl4579/StyleTTS2.git tmp_repo
!cp -rn tmp_repo/* StyleTTS2/ && rm -rf tmp_repo

%cd /kaggle/working/StyleTTS2
!pip install -q munch pydub pyyaml librosa nltk matplotlib accelerate transformers einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git

!python3 -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='yl4579/StyleTTS2-LibriTTS', local_dir='StyleTTS2-LibriTTS')"
!cp -rn StyleTTS2-LibriTTS/Models/* Models/
!ls -la Models/

/kaggle/working
Cloning into 'tmp_repo'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 372 (delta 200), reused 193 (delta 193), pack-reused 128 (from 1)
Receiving objects: 100% (372/372), 133.97 MiB | 31.47 MiB/s, done.
Resolving deltas: 100% (204/204), done.
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: cannot stat 'tmp_repo/*': No such file or directory
/kaggle/working/StyleTTS2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 4 files:   0%|                                   | 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (inc

In [5]:
%cd /kaggle/working
!rm -rf tmp_repo
!git clone https://github.com/yl4579/StyleTTS2.git tmp_repo
!cp -r --update=none tmp_repo/* StyleTTS2/
!rm -rf tmp_repo

%cd /kaggle/working/StyleTTS2
!mkdir -p Models/LJSpeech
!cp StyleTTS2-LibriTTS/epochs_2nd_00020.pth Models/LJSpeech/
!cp StyleTTS2-LibriTTS/config.yml Models/LJSpeech/
!ls -lh Configs/ Models/LJSpeech/ Data/

/kaggle/working
Cloning into 'tmp_repo'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 372 (delta 200), reused 193 (delta 193), pack-reused 128 (from 1)
Receiving objects: 100% (372/372), 133.97 MiB | 30.21 MiB/s, done.
Resolving deltas: 100% (204/204), done.
Updating files: 100% (48/48), done.
/kaggle/working/StyleTTS2
cp: cannot stat 'StyleTTS2-LibriTTS/epochs_2nd_00020.pth': No such file or directory
cp: cannot stat 'StyleTTS2-LibriTTS/config.yml': No such file or directory
Configs/:
total 12K
-rw-r--r-- 1 root root 3.4K Nov 25 23:48 config_ft.yml
-rw-r--r-- 1 root root 3.7K Nov 25 23:48 config_libritts.yml
-rw-r--r-- 1 root root 3.7K Nov 25 23:48 config.yml

Data/:
total 31M
-rw-r--r-- 1 root root  31M Nov 25 23:48 OOD_texts.txt
-rw-r--r-- 1 root root  25K Nov 25 23:47 train_list.txt
-rw-r--r-- 1 root root 2.8K Nov 25 23:47 val_list.txt
drwxr-xr-x 2 root root  12K Nov 25 23:47

In [6]:
path = "/kaggle/working/StyleTTS2/models.py"
with open(path, "r") as f:
    code = f.read()

# Fix the specific torch.load call failing on line 604
code = code.replace(
    "torch.load(model_path, map_location='cpu')",
    "torch.load(model_path, map_location='cpu', weights_only=False)"
)

with open(path, "w") as f:
    f.write(code)

In [7]:
import glob
import re

# Fix any `torch.load(weights_only=False, <rest>)` -> `torch.load(<rest>, weights_only=False)`
pattern = re.compile(r"torch\.load\(\s*weights_only\s*=\s*False\s*,\s*(.*?)\)\s*(?=#|\n|$)", re.DOTALL)

fixed_files = 0
for path in glob.glob("/kaggle/working/StyleTTS2/**/*.py", recursive=True):
    with open(path, "r", encoding="utf-8") as f:
        src = f.read()

    updated = pattern.sub(r"torch.load(\1, weights_only=False)", src)
    if updated != src:
        with open(path, "w", encoding="utf-8") as f:
            f.write(updated)
        fixed_files += 1

print(f"Fixed {fixed_files} file(s).")

# Verify line 30 of Utils/PLBERT/util.py
with open("/kaggle/working/StyleTTS2/Utils/PLBERT/util.py", "r", encoding="utf-8") as f:
    for idx, line in enumerate(f, 1):
        if 28 <= idx <= 32:
            print(f"{idx}: {line}", end="")

Fixed 0 file(s).
28:     iters = sorted(iters)[-1]
29: 
30:     checkpoint = torch.load(log_dir + "/step_" + str(iters) + ".t7", map_location='cpu')
31:     state_dict = checkpoint['net']
32:     from collections import OrderedDict


In [8]:
import re

path = "/kaggle/working/StyleTTS2/models.py"
with open(path, "r") as f:
    code = f.read()

# Clean up any partial fixes just in case, then append it correctly
code = code.replace(", weights_only=False", "") 
code = re.sub(r"(map_location=['\"]cpu['\"])", r"\1, weights_only=False", code)

with open(path, "w") as f:
    f.write(code)

print("models.py has been updated!")

models.py has been updated!
